In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate", "-q"])

import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
CUDA available: True
GPU count: 1
GPU 0: Tesla T4


In [3]:
DATASET_PATH = "/kaggle/input/datasets/sarveshwaran16/failurecase/dataset_clean.jsonl"

pairs = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Total pairs: {len(pairs)}")

Total pairs: 7024


In [4]:
def format_pair(pair):
    return {"text": f"### Instruction:\n{pair['instruction']}\n\n### Response:\n{pair['response']}"}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.08, seed=42)
print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

Train: 6462 | Test: 562


In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda:0",
)

model = prepare_model_for_kbit_training(model)
print("Model loaded!")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [6]:
# Expanded target_modules to include the MLP/feed-forward layers
# (gate_proj, up_proj, down_proj), not just attention (q/k/v/o_proj).
# Attention-only LoRA mostly shifts response STYLE (tone, brevity, format).
# The MLP layers are where a transformer stores more of its factual/domain
# knowledge -- adapting them gives LoRA real capacity to shift *what the
# model knows*, not just how it phrases things. This directly targets the
# factual-accuracy gaps seen in the previous run (e.g. PVC binding causes,
# CPU throttling diagnostics) rather than just adjusting surface style.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [7]:
from transformers import EarlyStoppingCallback, TrainerCallback
import time, os

# ---- Session safety net ----
# Your session is now ~9-12hrs, but "just extend the old 100min cap" is the
# wrong lever: the previous run's own logs show eval_loss bottoming out around
# step ~1248 (~3 epochs) and then getting WORSE every eval after that while
# train_loss kept dropping -- classic overfitting on a small (7024-pair)
# dataset. Early stopping (patience=2) was about to kill that run on its own
# right around where the old time cap also cut in. Simply letting it run
# longer under the same config would mostly extend the overfitting.
#
# So instead of deriving max_steps from a time probe, we now:
#   1. Train by epoch (num_train_epochs) with eval/save every epoch, so
#      early stopping gets a clean per-epoch signal instead of a noisy
#      mid-epoch one.
#   2. Add weight_decay + a cosine LR schedule to slow down overfitting,
#      so the model has a real chance to keep improving past epoch ~3.
#   3. Keep a hard wall-clock cap as a pure safety net (not the thing
#      driving training length) -- 330min (5.5hr) leaves ~30min buffer so
#      the whole run wraps inside a 6hr window, well short of the full
#      9-12hr session available.
#   4. Resume from the last checkpoint automatically if this cell reruns
#      after a session drop, so you don't lose progress.
TIME_BUDGET_MINUTES = 330  # hard safety-net cap only, not the training driver

class PlainTextLogger(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"[LOG] Training started. Total steps planned: {state.max_steps}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        elapsed = time.time() - self.start_time
        step = state.global_step
        loss = logs.get("loss")
        eval_loss = logs.get("eval_loss")
        msg = f"[LOG] step={step} elapsed={elapsed/60:.1f}min"
        if loss is not None:
            msg += f" train_loss={loss:.4f}"
        if eval_loss is not None:
            msg += f" eval_loss={eval_loss:.4f}"
        print(msg, flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"[LOG] Training finished at step {state.global_step}. Total time: {elapsed/60:.1f}min", flush=True)


class TimeBudgetCallback(TrainerCallback):
    """Pure safety net: force-stops training after a hard wall-clock limit
    regardless of epoch/step progress, in case something runs way slower
    than expected. Should not normally be the thing that ends training --
    early stopping (on eval_loss) is meant to do that."""
    def __init__(self, max_minutes):
        self.max_minutes = max_minutes
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed_min = (time.time() - self.start_time) / 60
        if elapsed_min >= self.max_minutes:
            print(f"[LOG] Safety time budget of {self.max_minutes}min reached at step {state.global_step} -- stopping now.", flush=True)
            control.should_training_stop = True
        return control


OUTPUT_DIR = "/kaggle/working/k8s-qwen"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,             # ceiling; early stopping will almost
                                      # certainly end things well before this
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    warmup_ratio=0.03,
    learning_rate=1e-4,              # lower than before (was 2e-4) to slow
                                      # the overfit seen in the last run
    weight_decay=0.01,               # regularization the previous run had none of
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=25,
    eval_strategy="epoch",           # was step-fraction; per-epoch is a
    save_strategy="epoch",           # cleaner signal for a small dataset
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),  # a bit more patient
                                                             # than before since
                                                             # each eval is now a
                                                             # full epoch, not a
                                                             # noisy mid-epoch point
        PlainTextLogger(),
        TimeBudgetCallback(max_minutes=TIME_BUDGET_MINUTES),
    ],
)

# Resume automatically if a checkpoint from an earlier (interrupted) session exists
resume_from = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_from = os.path.join(OUTPUT_DIR, sorted(ckpts, key=lambda d: int(d.split("-")[1]))[-1])
        print(f"[LOG] Found existing checkpoint, resuming from {resume_from}", flush=True)

print("Starting training...", flush=True)
trainer.train(resume_from_checkpoint=resume_from)
print("Training complete!", flush=True)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/6462 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6462 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/6462 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6462 [00:00<?, ? examples/s]

...

[LOG] step=1225 elapsed=65.8min train_loss=1.2629
[LOG] step=1250 elapsed=67.2min train_loss=1.1978
[LOG] step=1275 elapsed=68.5min train_loss=1.2071
[LOG] step=1300 elapsed=69.7min train_loss=1.1777
[LOG] step=1325 elapsed=71.1min train_loss=1.1809
[LOG] step=1350 elapsed=72.4min train_loss=1.1422
[LOG] step=1375 elapsed=73.7min train_loss=1.1633
[LOG] step=1400 elapsed=75.0min train_loss=1.1988
[LOG] step=1425 elapsed=76.3min train_loss=1.1940
[LOG] step=1450 elapsed=77.7min train_loss=1.2006
[LOG] step=1475 elapsed=79.0min train_loss=1.1667
[LOG] step=1500 elapsed=80.2min train_loss=1.2749
[LOG] step=1525 elapsed=81.5min train_loss=1.1821
[LOG] step=1550 elapsed=82.8min train_loss=1.2206
[LOG] step=1575 elapsed=84.2min train_loss=1.1559
[LOG] step=1600 elapsed=85.4min train_loss=1.2204
[LOG] step=1616 elapsed=86.9min eval_loss=1.5388


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[LOG] step=1625 elapsed=87.4min train_loss=1.0733
[LOG] step=1650 elapsed=88.7min train_loss=1.0152
[LOG] step=1675 elapsed=89.9min train_loss=1.0218
[LOG] step=1700 elapsed=91.3min train_loss=1.0352
[LOG] step=1725 elapsed=92.5min train_loss=1.0119
[LOG] step=1750 elapsed=93.9min train_loss=1.0232
[LOG] step=1775 elapsed=95.2min train_loss=0.9872
[LOG] step=1800 elapsed=96.5min train_loss=1.0539
[LOG] step=1825 elapsed=97.8min train_loss=1.0521
[LOG] step=1850 elapsed=99.1min train_loss=1.0487
[LOG] step=1875 elapsed=100.4min train_loss=0.9886
[LOG] step=1900 elapsed=101.7min train_loss=1.0649
[LOG] step=1925 elapsed=103.0min train_loss=1.0287
[LOG] step=1950 elapsed=104.2min train_loss=1.0451
[LOG] step=1975 elapsed=105.5min train_loss=1.1021
[LOG] step=2000 elapsed=106.9min train_loss=1.0548
[LOG] step=2020 elapsed=108.6min eval_loss=1.5858


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[LOG] step=2025 elapsed=108.8min train_loss=1.0203
[LOG] step=2050 elapsed=110.2min train_loss=0.8138
[LOG] step=2075 elapsed=111.4min train_loss=0.8671
[LOG] step=2100 elapsed=112.7min train_loss=0.8904
[LOG] step=2125 elapsed=114.0min train_loss=0.9026
[LOG] step=2150 elapsed=115.4min train_loss=0.9216
[LOG] step=2175 elapsed=116.8min train_loss=0.8657
[LOG] step=2200 elapsed=118.0min train_loss=0.8921
[LOG] step=2225 elapsed=119.3min train_loss=0.9214
[LOG] step=2250 elapsed=120.6min train_loss=0.8990
[LOG] step=2275 elapsed=121.9min train_loss=0.9265
[LOG] step=2300 elapsed=123.3min train_loss=0.8756
[LOG] step=2325 elapsed=124.6min train_loss=0.8823
[LOG] step=2350 elapsed=125.9min train_loss=0.9332
[LOG] step=2375 elapsed=127.2min train_loss=0.9386
[LOG] step=2400 elapsed=128.5min train_loss=0.8728
[LOG] step=2424 elapsed=130.3min eval_loss=1.6723
[LOG] step=2424 elapsed=130.3min
[LOG] Training finished at step 2424. Total time: 130.3min
Training complete!


In [ ]:
OUTPUT_PATH = "/kaggle/working/k8s-qwen-final"
trainer.model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Saved to /kaggle/working/k8s-tinyllama-final


In [9]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    device="cuda:0",
)

test_queries = [
    "Why is my pod in CrashLoopBackOff?",
    "What causes OOMKilled errors?",
    "Why is my PVC stuck in Pending?",
    "Why is my deployment continuously restarting?",
    "How can I investigate CPU throttling?",
    "What are common causes of 5XX errors in Kubernetes?",
]

for query in test_queries:
    prompt = f"### Instruction:\n{query}\n\n### Response:\n"
    result = pipe(prompt, do_sample=False)
    print(f"Q: {query}")
    print(f"A: {result[0]['generated_text'].replace(prompt, '').strip()[:250]}")
    print()


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take prece

Q: Why is my pod in CrashLoopBackOff?
A: Your pod is in CrashLoopBackOff because the container crashes repeatedly without completing its tasks.



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What causes OOMKilled errors?
A: OOMKilled errors occur when the kubelet kills a container due to resource exhaustion, such as memory or CPU.



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why is my PVC stuck in Pending?
A: Your PVC is stuck in Pending because the storage class 'default' has not been applied to your PVC. This can be resolved by applying the storage class using kubectl apply -f <file>.



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why is my deployment continuously restarting?
A: Your deployment is continuously restarting because the kubelet is constantly trying to start a new Pod, even if it has already started one. This can cause your deployment to restart itself multiple times due to its own logic and network issues.



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I investigate CPU throttling?
A: You can use the command `sudo top` to check CPU usage.

Q: What are common causes of 5XX errors in Kubernetes?
A: Common causes include: (1) the kubelet not being able to start, (2) a node or container restarting, (3) the kubelet failing to read its configuration file, and (4) the kubelet crashing.

